# RSIAT trên CUB-200 — tiếp tục từ `task_7.pth`

Bản notebook này đã sửa hai vấn đề trong file cũ:

1. Vá `timm==0.6.12` **ngay sau khi cài**, tránh lỗi `mutable default ... MaxxVitConvCfg`.
2. Tạo đúng cấu trúc dữ liệu CUB tại `data/datasets/cub/train` và `test`.

## Trước khi chạy

- Chọn **Runtime → Change runtime type → GPU**.
- Checkpoint được giả định nằm tại:
  `/content/drive/MyDrive/RSIAT_checkpoints/all/cub/10_10/seed_1993`
- Notebook cần bản `trainer.py` đã được bạn chỉnh để hỗ trợ resume.  
  Nó sẽ tự lấy từ `/content/drive/MyDrive/RSIAT_custom/trainer.py`; nếu không có, Colab sẽ yêu cầu bạn tải file lên.
- Chạy lần lượt từ trên xuống, không chạy lại cell cài `timm` sau cell vá.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Chưa bật GPU. Vào Runtime → Change runtime type → chọn GPU."
)

print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Google Drive và kiểm tra checkpoint

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_CKPT = Path(
    "/content/drive/MyDrive/RSIAT_checkpoints/"
    "all/cub/10_10/seed_1993"
)
TASK_TO_RESUME = 7
CKPT_NAME = f"task_{TASK_TO_RESUME}.pth"

assert DRIVE_CKPT.exists(), (
    f"Không tìm thấy thư mục checkpoint: {DRIVE_CKPT}"
)
assert (DRIVE_CKPT / CKPT_NAME).exists(), (
    f"Không tìm thấy {CKPT_NAME} trong {DRIVE_CKPT}"
)

print("Checkpoint trên Drive:")
for file in sorted(DRIVE_CKPT.glob("task_*.pth")):
    print(f"- {file.name}: {file.stat().st_size / 1024**2:.2f} MB")

## 3. Clone RSIAT và cài thư viện

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/RSIAT")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "https://github.com/zjrzjrz/RSIAT.git", str(REPO_DIR)],
    check=True,
)

print("Đã clone repo tại:", REPO_DIR)

In [ ]:
!pip install -q \
    easydict \
    timm==0.6.12 \
    optuna \
    umap-learn \
    pynndescent \
    scikit-learn \
    pandas \
    matplotlib \
    seaborn

## 4. Vá `timm 0.6.12` cho Python 3.12

In [ ]:
from pathlib import Path
import site

candidates = []
for base in site.getsitepackages():
    path = Path(base) / "timm" / "models" / "maxxvit.py"
    if path.exists():
        candidates.append(path)

assert candidates, "Không tìm thấy timm/models/maxxvit.py"

maxxvit_file = candidates[0]
text = maxxvit_file.read_text(encoding="utf-8")

# Vá theo cách idempotent: chạy lại cell cũng không tạo 'field, field'.
if "from dataclasses import dataclass, field" not in text:
    text = text.replace(
        "from dataclasses import dataclass",
        "from dataclasses import dataclass, field",
        1,
    )

old_conv = "conv_cfg: MaxxVitConvCfg = MaxxVitConvCfg()"
new_conv = "conv_cfg: MaxxVitConvCfg = field(default_factory=MaxxVitConvCfg)"
if old_conv in text:
    text = text.replace(old_conv, new_conv)

old_transformer = (
    "transformer_cfg: MaxxVitTransformerCfg = MaxxVitTransformerCfg()"
)
new_transformer = (
    "transformer_cfg: MaxxVitTransformerCfg = "
    "field(default_factory=MaxxVitTransformerCfg)"
)
if old_transformer in text:
    text = text.replace(old_transformer, new_transformer)

maxxvit_file.write_text(text, encoding="utf-8")

patched_text = maxxvit_file.read_text(encoding="utf-8")
assert new_conv in patched_text, "Vá conv_cfg chưa thành công"
assert new_transformer in patched_text, "Vá transformer_cfg chưa thành công"

print("Đã vá thành công:", maxxvit_file)

In [ ]:
import timm
import torch

print("timm:", timm.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert timm.__version__ == "0.6.12"
assert torch.cuda.is_available()

## 5. Cài bản `trainer.py` đã chỉnh resume

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/RSIAT")
DRIVE_TRAINER = Path(
    "/content/drive/MyDrive/RSIAT_custom/trainer.py"
)
TARGET_TRAINER = REPO_DIR / "trainer.py"

if DRIVE_TRAINER.exists():
    shutil.copy2(DRIVE_TRAINER, TARGET_TRAINER)
    print("Đã lấy trainer.py từ Drive:", DRIVE_TRAINER)
else:
    from google.colab import files

    print(
        "Không tìm thấy trainer.py tại RSIAT_custom trên Drive. "
        "Hãy chọn bản trainer.py đã chỉnh resume."
    )
    uploaded = files.upload()

    assert "trainer.py" in uploaded, (
        "Bạn cần tải lên đúng file có tên trainer.py"
    )
    TARGET_TRAINER.write_bytes(uploaded["trainer.py"])
    print("Đã chép trainer.py vào:", TARGET_TRAINER)

trainer_text = TARGET_TRAINER.read_text(encoding="utf-8")
print("Kích thước trainer.py:", len(trainer_text), "ký tự")
print("Có từ khóa resume:", "resume" in trainer_text.lower())
print("Có torch.load:", "torch.load" in trainer_text)

## 6. Tạo lại `adapter_cub.json`

In [ ]:
import json
from pathlib import Path

CONFIG = {'prefix': 'all', 'dataset': 'cub', 'shuffle': True, 'ssca': True, 'ca': True, 'init_cls': 10, 'increment': 10, 'model_name': 'adapter', 'convnet_type': 'pretrained_vit_b16_224_in21k_adapter', 'device': ['0'], 'seed': [1993], 'init_epochs': 20, 'warmup_epoch': 9, 'inc_epochs': 30, 'ca_epochs': 10, 'init_lr': 0.03773856474362839, 'batch_size': 128, 'weight_decay': 3.425582163491081e-06, 'min_lr': 0, 'ffn_num': 64, 'optimizer': 'sgd', 'scale': 20.0, 'margin': 0.1, 'alpha': 1.0, 'rs_margin': 0.5, 'lambda_rs': 0.2, 'beta': 1.5, 'gamma': 0.75, 'ae_code_dims': 768, 'ae_init_lr': 0.09988605662355629, 'ae_weight_decay': 9.418584598386518e-05, 'resume': True, 'resume_path': ''}

CONFIG_FILE = Path("/content/RSIAT/exps/adapter_cub.json")
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.write_text(
    json.dumps(CONFIG, indent=4),
    encoding="utf-8",
)

print("Đã ghi config:", CONFIG_FILE)
print(CONFIG_FILE.read_text(encoding="utf-8"))

## 7. Tải và giải nén CUB-200-2011

In [ ]:
from pathlib import Path
import subprocess

ARCHIVE = Path("/content/CUB_200_2011.tgz")
CUB_ROOT = Path("/content/CUB_200_2011")
CUB_URL = (
    "https://data.caltech.edu/records/65de6-vp158/"
    "files/CUB_200_2011.tgz?download=1"
)

if not CUB_ROOT.exists():
    if not ARCHIVE.exists():
        subprocess.run(
            ["wget", "-O", str(ARCHIVE), CUB_URL],
            check=True,
        )

    subprocess.run(
        ["tar", "-xzf", str(ARCHIVE), "-C", "/content"],
        check=True,
    )
else:
    print("Dữ liệu đã được giải nén, bỏ qua bước tải.")

required_files = [
    CUB_ROOT / "images",
    CUB_ROOT / "images.txt",
    CUB_ROOT / "train_test_split.txt",
]
for path in required_files:
    assert path.exists(), f"Thiếu dữ liệu: {path}"

print("CUB root:", CUB_ROOT)

## 8. Tạo cấu trúc `train/test` mà RSIAT yêu cầu

In [ ]:
from pathlib import Path
import shutil

CUB_ROOT = Path("/content/CUB_200_2011")
OUTPUT_ROOT = Path("/content/RSIAT/data/datasets/cub")
TRAIN_DIR = OUTPUT_ROOT / "train"
TEST_DIR = OUTPUT_ROOT / "test"

EXPECTED_TRAIN_IMAGES = 5994
EXPECTED_TEST_IMAGES = 5794
EXPECTED_CLASSES = 200

def count_images(directory):
    return sum(1 for p in directory.rglob("*") if p.is_file())

dataset_ready = (
    TRAIN_DIR.exists()
    and TEST_DIR.exists()
    and len([p for p in TRAIN_DIR.iterdir() if p.is_dir()]) == EXPECTED_CLASSES
    and len([p for p in TEST_DIR.iterdir() if p.is_dir()]) == EXPECTED_CLASSES
    and count_images(TRAIN_DIR) == EXPECTED_TRAIN_IMAGES
    and count_images(TEST_DIR) == EXPECTED_TEST_IMAGES
)

if not dataset_ready:
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

    image_paths = {}
    with (CUB_ROOT / "images.txt").open("r", encoding="utf-8") as f:
        for line in f:
            image_id, relative_path = line.strip().split(maxsplit=1)
            image_paths[image_id] = relative_path

    split_map = {}
    with (CUB_ROOT / "train_test_split.txt").open(
        "r", encoding="utf-8"
    ) as f:
        for line in f:
            image_id, is_train = line.strip().split()
            split_map[image_id] = is_train == "1"

    assert image_paths.keys() == split_map.keys(), (
        "images.txt và train_test_split.txt không khớp ID."
    )

    for image_id, relative_path in image_paths.items():
        source = CUB_ROOT / "images" / relative_path
        split_name = "train" if split_map[image_id] else "test"
        destination = OUTPUT_ROOT / split_name / relative_path

        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
else:
    print("Dataset train/test đã đúng, bỏ qua bước sao chép.")

train_classes = len([p for p in TRAIN_DIR.iterdir() if p.is_dir()])
test_classes = len([p for p in TEST_DIR.iterdir() if p.is_dir()])
train_images = count_images(TRAIN_DIR)
test_images = count_images(TEST_DIR)

print("Train classes:", train_classes)
print("Train images:", train_images)
print("Test classes:", test_classes)
print("Test images:", test_images)

assert train_classes == EXPECTED_CLASSES
assert test_classes == EXPECTED_CLASSES
assert train_images == EXPECTED_TRAIN_IMAGES
assert test_images == EXPECTED_TEST_IMAGES

print("Dataset CUB đã sẵn sàng tại:", OUTPUT_ROOT)

## 9. Liên kết log và checkpoint với Google Drive

In [ ]:
from pathlib import Path
import os
import shutil

REPO_DIR = Path("/content/RSIAT")

# Log được ghi trực tiếp lên Drive.
DRIVE_LOGS = Path("/content/drive/MyDrive/RSIAT_logs")
LOCAL_LOGS = REPO_DIR / "logs"
DRIVE_LOGS.mkdir(parents=True, exist_ok=True)

if LOCAL_LOGS.exists() or LOCAL_LOGS.is_symlink():
    if LOCAL_LOGS.is_symlink() or LOCAL_LOGS.is_file():
        LOCAL_LOGS.unlink()
    else:
        shutil.rmtree(LOCAL_LOGS)

os.symlink(DRIVE_LOGS, LOCAL_LOGS, target_is_directory=True)

# Checkpoint local trỏ thẳng tới thư mục checkpoint trên Drive.
LOCAL_CKPT = (
    REPO_DIR
    / "checkpoints"
    / "all"
    / "cub"
    / "10_10"
    / "seed_1993"
)
LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_CKPT.exists() or LOCAL_CKPT.is_symlink():
    if LOCAL_CKPT.is_symlink() or LOCAL_CKPT.is_file():
        LOCAL_CKPT.unlink()
    else:
        shutil.rmtree(LOCAL_CKPT)

os.symlink(DRIVE_CKPT, LOCAL_CKPT, target_is_directory=True)

print("Logs:")
print(" ", LOCAL_LOGS, "->", os.path.realpath(LOCAL_LOGS))
print("Checkpoint:")
print(" ", LOCAL_CKPT, "->", os.path.realpath(LOCAL_CKPT))
print("Checkpoint cần resume:", LOCAL_CKPT / CKPT_NAME)

assert (LOCAL_CKPT / CKPT_NAME).exists()

## 10. Kiểm tra đọc `task_7.pth` trước khi chạy

In [ ]:
import gc
import os
import sys
from pathlib import Path

os.chdir("/content/RSIAT")
if "/content/RSIAT" not in sys.path:
    sys.path.insert(0, "/content/RSIAT")

import torch

CKPT_FILE = Path(
    "/content/RSIAT/checkpoints/all/cub/10_10/"
    f"seed_1993/{CKPT_NAME}"
)

print("Checkpoint:", CKPT_FILE)
print(f"Size: {CKPT_FILE.stat().st_size / 1024**2:.2f} MB")

checkpoint = torch.load(
    CKPT_FILE,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint đọc thành công.")
if isinstance(checkpoint, dict):
    print("Keys:", list(checkpoint.keys()))
else:
    print("Kiểu dữ liệu:", type(checkpoint))

del checkpoint
gc.collect()

## 11. Tiếp tục huấn luyện

Khi resume hoạt động đúng, chương trình phải bỏ qua các task đã hoàn thành và tiếp tục từ task tiếp theo sau `task_7.pth`.

**Không đóng tab Colab trong lúc chạy.** Checkpoint và log mới sẽ được ghi trực tiếp lên Google Drive.

In [ ]:
%cd /content/RSIAT

!python -u main.py --config ./exps/adapter_cub.json

## 12. Kiểm tra checkpoint sau khi chạy

In [ ]:
from pathlib import Path

print("Checkpoint hiện có trên Drive:")
for file in sorted(DRIVE_CKPT.glob("task_*.pth")):
    print(f"- {file.name}: {file.stat().st_size / 1024**2:.2f} MB")